In [ ]:
import pandas as pd
import numpy as np

from scipy import stats
from statsmodels.stats.multitest import multipletests
import statsmodels.formula.api as smf
import statsmodels.api as sm

import matplotlib as mpl
from matplotlib import pyplot as plt
from matplotlib.patches import Arc, FancyArrow
from matplotlib.lines import Line2D
from matplotlib import cm, colors, colormaps
from matplotlib.colors import ListedColormap
import colorsys

import seaborn as sns
import colorcet
from phytreeviz import TreeViz
# Dunno why phytreeviz changes these at import, took me forever to find it. Revert back to values.
mpl_rc_params = {
    "svg.fonttype": "path",
    "savefig.bbox": "standard",
}
mpl.rcParams.update(mpl_rc_params)

import itertools
from collections import Counter
import re

from Bio import SeqIO
from Bio import SeqRecord
from Bio import Seq
from Bio import Phylo
import allel

import zarr
import sys
from progressbar import ProgressBar
from collections import Counter

import mathieu as mh

# Define general variables

In [ ]:
# Set paths
base_dir = '/home/mathieu/postdoc_heasley/short_read_project/'
ref_dir = '/home/mathieu/postdoc_heasley/short_read_project/data/ref/'
fig_path = f'/home/mathieu/postdoc_heasley/publications/subtelomere_paper/fig/'

In [ ]:
# Import strains metadata tables
strains_info = pd.read_csv(f'{base_dir}script/strains_info.csv').set_index('strain', drop=False)
collection = pd.read_excel('/home/mathieu/postdoc_heasley/collection/mccusker_collection_wild_annot.xlsx').set_index('ID', drop=False)
wine_subclade = [f'YJM{i}' for i in (947, 948, 954, 955, 956, 957, 963, 964, 965, 967)]
wine_subclade_order = list(collection.loc[wine_subclade, 'Y\' element'].sort_values().index)
strain_yprime_color = dict(zip(wine_subclade_order, [colormaps.get_cmap('viridis')(i) for i in np.linspace(0,1,10)]))

In [ ]:
# Isolation source metadata and annotation
source_order = strains_info.loc[strains_info['species']=='cerevisiae', 'source'].value_counts().index

source_alias = dict(zip(source_order, [x[0].upper()+x[1:].replace('_', ' ') for x in source_order]))
source_color = dict(zip(source_order, [c+[0.8] for c in colorcet.glasbey_bw[:source_order.shape[0]]]))
Source_color = {source_alias[i]:j for (i,j) in source_color.items()}
source_idx = dict(zip(source_order, range(len(source_order))))

source_order_dict = dict(zip(source_order, range(len(source_order))))
cmap_source_color = colors.LinearSegmentedColormap.from_list('source_color', colorcet.glasbey_bw[:source_order.shape[0]], N=source_order.shape[0])

source_simplified_dict = {'clinical':0,
                          'wine':1,
                          'fruit':2,
                          'fermented_foods':1,
                          'brewing':1,
                          'unknown':3,
                          'fermented_beverages':1,
                          'soil':2,
                          'industry':1,
                          'tree':2,
                          'sake':1,
                          'insect':2,
                          'distillery':1,
                          'dairy':1,
                          'baking':1,
                          'bark':2,
                          'human':1,
                          'palm_nectar':2,
                          'commercial':1,
                          'sewage':1, 
                          'molasses':1,
                          'natural_fermentation':1,
                          'grain':1,
                          'water':1, 
                          'evolution_canyon':2, 
                          'plants':2,
                          'rotten_wood':2,
                          'mushrooms':2,
                          'spoiled_foods':1,
                          'fermentation':1,
                          'animal':1,
                          'alpechin':1,
                          'vegetables':1,
                          'slime_flux':2,
                          'flower':2,
                          'honey':2}

source_simplified_order = ['clinical', 'domesticated', 'environmental', 'unknown']
source_simplified_color = dict(zip(source_simplified_order, ['red', 'royalblue', 'limegreen', '0.7']))
cmap_source_simplified_color = colors.LinearSegmentedColormap.from_list('source_simplified_color', ['red', 'royalblue', 'limegreen', '0.7'], N=4)

## Reference genome annotations

In [ ]:
chr_list = pd.read_csv(f'{ref_dir}S288C.chr_list.tsv', sep='\t')
chr_ctg_to_alias = chr_list.set_index('contig_name')['alias'].to_dict()
chr_alias_to_ctg = {j:i for (i,j) in chr_ctg_to_alias.items()}

with open(f'{ref_dir}S288C.fasta') as fi:
    ref_genome = [seq for seq in SeqIO.parse(fi, 'fasta')]

nuclear_chromosomes = [seq.id for seq in ref_genome][:-1]

with open(f'{ref_dir}S288C_masked.repeats.fasta') as fi:
    ref_genome_masked = [seq for seq in SeqIO.parse(fi, 'fasta')]
repeats = [seq.id for seq in ref_genome_masked if seq.id not in nuclear_chromosomes]

repeats_alias = {'ref|NC_001224|':'mtDNA',
                 'rDNA.cons_trim':'rDNA',
                 'X_element.cons_trim':'X element',
                 'Y_prime_element.cons_trim':'Y\' element',                
                 'TY1-I#LTR/Copia':'Ty1-I',
                 'TY1-LTR#LTR/Copia':'Ty1-LTR',
                 'TY2-I#LTR/Copia':'Ty2-I',
                 'TY2-LTR#LTR/Copia':'Ty2-LTR',
                 'TY3-I#LTR/Gypsy':'Ty3-I',
                 'TY3-LTR#LTR/Gypsy':'Ty3-LTR',
                 'TY3_1p-I#LTR/Gypsy':'Ty3p-I',
                 'TY3_1p-LTR#LTR/Gypsy':'Ty3p-LTR',
                 'TY4-I#LTR/Copia':'Ty4-I',
                 'TY4-LTR#LTR/Copia':'Ty4-LTR',
                 'TY5-I#LTR/Copia':'Ty5-I',
                 'TY5-LTR#LTR/Copia':'Ty5-LTR'               
                }
repeats_order = list(repeats_alias.keys())

## Reference genome annotations

In [ ]:
# Generate gff file from SGD reference, without header, without fasta, converting contig names.

new_gff = []
with open('/home/mathieu/postdoc_heasley/sequences/reference_genomes/S288C/saccharomyces_cerevisiae_R64-4-1_20230830.gff') as fi:
    for l in fi.read().splitlines():
        if l[0] == '#':
            include = True
        elif l[:3] == 'chr':
            l = l.split('\t')
            l[0] = chr_alias_to_ctg[l[0]]
            l = '\t'.join(l)
            include = True
        else:
            include = False
        if include:
            new_gff.append(l)

#with open('/home/mathieu/postdoc_heasley/sequences/reference_genomes/S288C/S288C_current_reference.gff', 'w') as fo:
#    fo.write('\n'.join(new_gff))
del new_gff

In [ ]:
gff = pd.read_csv(f'{ref_dir}S288C.repeats.gff', sep='\t', comment='#', header=None)
bed = pd.read_csv(f'{ref_dir}S288C.bed', sep='\t', comment='#', header=None)

centromeres = gff.loc[gff[2]=='centromere'].groupby(0).apply(lambda x: x[[3,4]].values.flatten().mean())

In [ ]:
# Convert and export GFF to BED

gff_annot_exploded = pd.concat([gff.iloc[:, :8], gff[8].apply(lambda x: pd.Series(dict([i.split('=') for i in x.split(';')])))], axis=1)

gff_annot_exploded['bed_name'] = ''
for at, df in gff_annot_exploded.groupby(2):
    df = df.iloc[:, 8:].dropna(axis=1, how='any')
    gff_annot_exploded.loc[df.index, 'bed_name'] = df.apply(lambda x: ' '.join(x.astype(str))\
                                                            .replace('%20', ' ').replace('%3B', ';').replace('%2C', ',')\
                                                            .replace('%28', '(').replace('%29', ')').strip(), axis=1)

new_bed = gff_annot_exploded[[0, 3, 4, 'bed_name', 5, 6]].copy()
new_bed[3] -= 1
new_bed[4] -= 1

#new_bed.to_csv('/home/mathieu/postdoc_heasley/sequences/reference_genomes/S288C/S288C_current_reference.bed', sep='\t', header=False, index=False)

In [ ]:
# Genome display variables
chrom_order = list(chr_list['alias'])
ref_chr_len = [len(seq.seq) for seq in ref_genome]
ref_chr_len_dict = dict(zip(chrom_order, ref_chr_len))

tig_off = pd.Series(ref_chr_len_dict, name='len')
tig_off = pd.concat([tig_off, tig_off.cumsum().rename('cumul_end')], axis=1)
tig_off['cumul_start'] = np.concatenate([[0], tig_off['len'].iloc[:-1].cumsum().values])

## 1011 genomes metadata

In [ ]:
strains_1011 = pd.read_csv(f'{base_dir}script/1011_genomes/peter_nature2018_S1.tsv', sep='\t').set_index('Standardized name', drop=False)
# Annotate clades
strains_1011['Clades_original'] = strains_1011['Clades'].fillna('Unclustered').apply(lambda x: x.strip(' '))
strains_1011['Clades'] = strains_1011['Clades'].fillna('Unclustered').apply(lambda x: x.strip(' '))
# Manually erase the subclades of Wine/European
strains_1011['Clades'] = strains_1011['Clades'].replace({f'1. Wine/European (subclade {i})':'1. Wine/European' for i in range(1,5)})
pattern_clade_no = r'(\d+)\. \w+'
clade_no_unmatched = {'M1. Mosaic region 1': 27, 'M2. Mosaic region 2':28, 'M3. Mosaic region 3':29, 'Unclustered': 30}
strains_1011['clade_no'] = strains_1011['Clades'].apply(lambda x: int(re.match(pattern_clade_no, x).group(1)) if re.match(pattern_clade_no, x) else clade_no_unmatched[x])
strains_1011['Total number of SNPs'] = strains_1011['Total number of SNPs'].apply(lambda x: x.replace(',', '')).astype(np.int64)
strains_1011['Het_Freq'] = strains_1011['Total number of SNPs']*strains_1011['Proportion of clean heterozygous SNPs (whole dataset)']*0.01/sum(ref_chr_len)

# Get order of the random subset
strains_1011_order = list(strains_1011.loc[strains_info.loc[~strains_info['sra'].isna()].index].sort_values(by=['clade_no', 'Standardized name']).index)

In [ ]:
clades_unique = strains_1011.loc[strains_1011_order].groupby('clade_no').apply(lambda x: x['Clades'].iloc[0], include_groups=False).rename('Clades')
clades_unique.loc[1] = '1. Wine/European'
clades_unique.loc[30] = 'Soil/tree'
clades_unique.loc[31] = 'Californian sour figs'
clades_unique.loc[32] = 'Unclustered'

clades_unique = clades_unique.reset_index()
clades_unique.index = clades_unique['Clades'].values
# Manually populate with the new YJM strains
clades_unique['mfc'] = [colors.rgb2hex(c) for c in colorcet.glasbey_hv[:clades_unique.shape[0]]]
clades_unique['mec'] = clades_unique['mfc']
clades_unique.loc['Unclustered', ['mfc', 'mec']] = ('#ffffff', '#000000')

In [ ]:
#clades_unique.to_csv(f'{base_dir}script/clades_unique.csv')
clades_unique = pd.read_csv(f'{base_dir}script/clades_unique.csv', index_col=0)

In [ ]:
# Define clade metadata and annotation from the 1011 genomes paper
clade_order = clades_unique.index
clade_color = clades_unique['mfc'].to_dict()
clade_idx = dict(zip(clade_order, range(len(clade_order))))
strains_info['clade_idx'] = strains_info['clade'].apply(lambda x: clade_idx.get(x, np.nan))

# Define general plot annotation functions

In [ ]:
def make_listedcolormap(colors):
    N = len(colors)
    cmap = colors.ListedColormap(colors)
    bounds = np.arange(N+1)
    norm = colors.BoundaryNorm(bounds, cmap.N)
    return(cmap, norm)

In [ ]:
def plot_cbar_vertical(hm_data, yticks, ytick_labels, title, width=0.01, height=0.15, top_padding=1.05, fontsize=8, x=None, y=None):
    
    global fig
    global ax
    
    if x == None and y == None:
        (xmin, ymin), (xmax, ymax) = np.array(ax.get_position())
        x = np.mean(np.array([xmin, xmax])-0.5*width)
        y = ymax*top_padding
    
    ax_cbar = fig.add_axes([x, y, width, height])
    
    cb = plt.colorbar(hm_data, orientation='vertical', cax=ax_cbar)
    cb.outline.set_visible(False)
    
    ax_cbar.text(-0.1, 0.5, title, rotation=90, size=fontsize, 
                 ha='right', va='center', transform=ax_cbar.transAxes)
    ax_cbar.set_yticks(yticks)
    ax_cbar.set_yticklabels(ytick_labels, size=fontsize)

In [ ]:
collection.loc[wine_subclade, 'Y\' element'].apply(np.round)

# YJM collection numbers

In [ ]:
f_strains_dup = ~strains_info['is_duplicate']
f_strains_yjm = strains_info['dataset']!='1011_genomes'
f_strains_cer = strains_info['species']=='cerevisiae'


fig = plt.figure(figsize=[5.5, 2])
gs = plt.GridSpec(ncols=1, nrows=2, hspace=1.8, left=0.05, top=0.75, bottom=0.22, right=0.67)

# Isolation source of the whole collection

ax = fig.add_subplot(gs[0])
F = f_strains_dup & f_strains_yjm
dat = strains_info.loc[F].value_counts('source_simplified').sort_values(ascending=False)
dat = pd.concat([dat, (dat.cumsum()/dat.sum()*100).rename('cumul')], axis=1)

for ss, (ct, cs) in dat[::-1].iterrows():
    ax.barh([0], [cs], color=source_simplified_color[ss])
    if ss in ('environmental', 'unknown'):
        c = 'k'
    else:
        c = 'w'
    ax.text(cs, 0, int(ct), size=8, color=c, va='center', ha='right')

for sp in ['top', 'left', 'right']:
    ax.spines[sp].set_visible(False)
ax.set_xlim(0, 100)
ax.set_yticks([])
ax.set_title('All isolates - sampling source', size=11, loc='left', ha='left')

legend_elms = [Line2D([0], [0], color='w', marker='s', ms=12, label=label, mfc=ss, mew=0)
               for label, ss in source_simplified_color.items()]

ax.legend(handles=legend_elms, bbox_to_anchor=[1.05, 0.5], loc=6, frameon=False, fontsize=10)

# Sequencing status of the cerevisiae

ax = fig.add_subplot(gs[1])

F = f_strains_dup & f_strains_yjm & f_strains_cer
dat = strains_info.loc[F, ['seq_before','seq_spore']].replace({'seq_before':{False:0.1, True:0.9}, 'seq_spore':{False:0, True:-0.4}})\
    .sum(axis=1).value_counts().sort_index()
dat = pd.concat([dat, (dat.cumsum()/dat.sum()*100).rename('cumul')], axis=1)

for sb, (ct, cs) in dat[::-1].iterrows():
    ax.barh([0], [cs], color=plt.get_cmap('binary')(sb))
    if sb == 0.1:
        c = 'k'
    else:
        c = 'w'
    ax.text(cs, 0, int(ct), size=8, color=c, va='center', ha='right')

for sp in ['top', 'left', 'right']:
    ax.spines[sp].set_visible(False)
ax.set_xlim(0, 100)
ax.set_xlabel('% isolates')
ax.set_yticks([])
ax.set_title('$S. cerevisiae$ isolates - previous sequencing', size=11, loc='left', ha='left')

legend_elms = [Line2D([0], [0], color='w', marker='s', ms=12, label=label, mfc=plt.get_cmap('binary')(sb), mew=0) 
               for sb, label in zip([0.1, 0.5, 0.9], ['Never', 'As spore', 'Native'])]
ax.legend(handles=legend_elms, bbox_to_anchor=[1.05, 0.5], loc=6, frameon=False, fontsize=10)

    
plt.show()
plt.close()

# Clinical isolates are overrepresented in some clades

In [ ]:
def make_contingency_table(df, v1, v2):
    ct = df.value_counts([v1, v2]).reset_index()\
    .pivot_table(index=v1, columns=v2, values='count').fillna(0)
    return ct

In [ ]:
clade_source = strains_info.loc[f_strains_cer & f_strains_dup & f_strains_yjm, ['clade', 'source_simplified']].copy()
contingency_table = clade_source.value_counts(['source_simplified','clade']).reset_index()\
    .pivot_table(index='clade', columns='source_simplified', values='count').fillna(0)
contingency_table = contingency_table.loc[[c for c in clade_order if c in contingency_table.index]]
contingency_table['total'] = contingency_table.sum(axis=1)

frac_per_clade = contingency_table['total']/contingency_table['total'].sum()
count_per_source = contingency_table[source_simplified_order].sum()
expected = np.tile(frac_per_clade.values.reshape(-1, 1), 4)
expected = np.apply_along_axis(lambda x: x*count_per_source, 1, expected)
expected = pd.DataFrame(expected, columns=source_simplified_order, index=contingency_table.index)

### Fig S1BC

In [ ]:
fig = plt.figure(figsize=[10, 5])
gs = plt.GridSpec(ncols=1, nrows=2, hspace=0.3, height_ratios=[3,4],
                 left=0.15, top=0.94, right=1, bottom=0.42)

ax = fig.add_subplot(gs[0])

dat = contingency_table.T.loc[source_simplified_order[:-1]]

sns.heatmap(dat, cmap='viridis', #center=1, vmin=-25, vmax=25, ax=ax,
            cbar_kws={'aspect':8, 'label':'# isolates'}, ax=ax)

X = np.arange(dat.shape[1])+0.5
ax.set_xlabel('')
ax.set_xticklabels([])
ax.scatter(X, np.repeat(3.3, X.shape[0]), clip_on=False, s=60, linewidths=1,
          c=[clade_color[c] for c in dat.columns], 
          edgecolors=['k' if c in ['Unclustered', '23. North American oak'] else 'w' for c in dat.columns])

ax.set_ylabel('')
ax.set_yticks(np.arange(3)+0.5)
ax.set_yticklabels(source_simplified_order[:-1], rotation=0)


ax = fig.add_subplot(gs[1])

dat = (contingency_table[source_simplified_order]-expected).T.loc[source_simplified_order[:-1]]

chi_square_clade_source = {}
for s in source_simplified_order[:-1]:
    e = expected[s]
    chi, pval = stats.chisquare(contingency_table[s], e)
    chi_square_clade_source[s] = pval

sns.heatmap(dat, cmap='PiYG', center=1, vmin=-25, vmax=25, ax=ax,
            cbar_kws={'aspect':8, 'ticks':(-25,0,25), 'label':'enrichment\n(# isolates)'})

ax.set_ylabel('')
ax.set_yticks(np.arange(3)+0.5)
ax.set_yticklabels([f'{s}\n'+f'p={mh.plot_pval_text(chi_square_clade_source[s])}' 
                    for s in source_simplified_order[:-1]], rotation=0)

ax.set_xlabel('')
X = np.arange(dat.shape[1])+0.5

ax.set_xticks(X)
ax.set_xticklabels(dat.columns, rotation=90)
for xt in ax.get_xticklabels():
    clade = xt.get_text()
    c = clade_color[clade]
    r, g, b = colors.to_rgb(c)
    h, l, s = colorsys.rgb_to_hls(r, g, b)
    
    xt.set_backgroundcolor(c)
    xt.get_bbox_patch().set_boxstyle('square', pad=0.15)

    if l < 0.5:
        xt.set_color('w')

fig.text(0.01, 0.93, 'B', size=24, fontweight='bold')
fig.text(0.01, 0.68, 'C', size=24, fontweight='bold')

for ext in ['png', 'svg']:
    plt.savefig(f'{fig_path}FigS1BC.{ext}', dpi=300)

plt.show()
plt.close()

# VCF preprocessing

In [ ]:
# Export a 1-based bed file excluding repetitive, subtelomeric, rDNA, mating-type region loci
exclude_chromosome_ends = pd.read_csv(f'{base_dir}script/S288C_exclude_chromosome_ends.tsv', sep='\t', comment='#', header=None)
feature_exclude = ['long_terminal_repeat', 'transposable_element_gene', 'telomere', \
                   'LTR_retrotransposon', 'X_element', 'telomeric_repeat', 'X_element_combinatorial_repeat', 'Y_prime_element', \
                   'mating_type_region', 'rRNA_gene', 'rRNA', 'external_transcribed_spacer_region', 'internal_transcribed_spacer_region']

exclude = pd.concat([gff.loc[gff[2].isin(feature_exclude), [0,3,4]].rename({3:1, 4:2}, axis=1), exclude_chromosome_ends[[0,1,2]]]).reset_index(drop=True)
exclude = exclude.astype({0:str, 1:np.int64, 2:np.int64}).sort_values(by=[0,1,2])
#exclude.to_csv(f'{base_dir}script/S288C_exclude.tab', sep='\t', index=False, header=False)

# Divergence and hetrozygosity levels

In [ ]:
vcf_path_s = f'{base_dir}mpileup_all/mpileup_snps.add_tags.fil.corr.vcf.gz'
zarr_path_s = f'{base_dir}mpileup_all/mpileup_snps.add_tags.fil.corr.zarr'
#allel.vcf_to_zarr(vcf_path_s, zarr_path_s, fields='*')

In [ ]:
#callset_s = zarr.open_group(zarr_path_s, mode='r')
#gt_s = allel.GenotypeChunkedArray(callset_s['/calldata']['GT'])
callset_s = allel.read_vcf(vcf_path_s)
gt_s = allel.GenotypeChunkedArray(callset_s['calldata/GT'])

In [ ]:
CHROM_s = callset_s['/variants']['CHROM'][:]
POS_s = callset_s['/variants']['POS'][:]
Pos_Bin_s = pd.cut(POS_s, pd.interval_range(start=0, end=1.6e6, freq=1e4, closed='left'))
pos_bin_s = np.array([i.mid for i in Pos_Bin_s])

In [ ]:
vcf_samples_s = pd.Series(callset_s['samples'][:])
vcf_samples_s_idx = pd.Series(np.arange(vcf_samples_s.shape[0]), index=vcf_samples_s)

In [ ]:
vcf_path_1011 = f'{base_dir}script/1011_genomes/1011Matrix.gvcf.gz'
zarr_path_1011 = f'{base_dir}script/1011_genomes/1011_genomes.zarr'
#allel.vcf_to_zarr(vcf_path_1011, zarr_path_1011, fields='*')

In [ ]:
callset_1011 = zarr.open_group(zarr_path_1011)
gt_1011 = allel.GenotypeChunkedArray(callset_1011['/calldata']['GT'])

In [ ]:
vcf_samples_1011 = pd.Series(callset_1011['samples'][:])
vcf_samples_1011_idx = pd.Series(np.arange(vcf_samples_1011.shape[0]), index=vcf_samples_1011)

## Allele depth

In [ ]:
gt_ad = allel.GenotypeChunkedArray(callset_s['/calldata']['AD'])

In [ ]:
# replace negative values by 0 for sum computation
gt_ad_z = np.sort(gt_ad[:], axis=2)[:,:,::-1]
gt_ad_z[gt_ad_z<1] = 0

In [ ]:
maj_ad = gt_ad_z[:,:,0]
min_ad = gt_ad_z[:,:,1]
sum_ad = gt_ad_z.sum(axis=2)

In [ ]:
maf = maj_ad/sum_ad

### MAF vs AD

In [ ]:
min_ad_het = min_ad[gt_s_het]
min_ad_hom = min_ad[~gt_s_het]

In [ ]:
fig = plt.figure(figsize=[12,7])
gs = plt.GridSpec(ncols=2, nrows=2)


t = 12

ax = fig.add_subplot(gs[0, 0])
ax.hist(maf[(gt_s_het) & (min_ad>=t)], bins=np.linspace(0,1,50))
ax.set_title(f'above t={t}')

ax = fig.add_subplot(gs[1, 0])
ax.hist(maf[(gt_s_het) & (min_ad<t)], bins=np.linspace(0,1,50))
ax.set_title(f'under t={t}')

ax = fig.add_subplot(gs[:, 1])
h2d = np.histogram2d(min_ad[gt_s_het], maf[gt_s_het], bins=[np.linspace(0, 175, 176), np.linspace(0.5, 1, 200)])


ax.imshow(h2d[0], vmin=0, vmax=np.quantile(h2d[0], 0.98), 
          cmap='viridis', aspect='auto', interpolation='nearest')
ax.invert_yaxis()

xticks = np.arange(0, h2d[2].shape[0], 20)
ax.set_xticks(xticks)
ax.set_xticklabels([f'{h2d[2][i]:.2f}' for i in xticks], rotation=90)

ax.axhline(t, color='red', lw=0.5)

plt.show()
plt.close()

## Heterozygosity

In [ ]:
gt_s_het = gt_s.is_het()[:]

In [ ]:
het_freq = (gt_s_het.sum(axis=0)/sum(ref_chr_len))[:]
het_freq = pd.Series(het_freq, index=vcf_samples_s)

In [ ]:
gt_1011_het = gt_1011.is_het()
het_freq_1011 = (gt_1011_het.sum(axis=0)/sum(ref_chr_len))[:]
het_freq_1011 = pd.Series(het_freq_1011, index=vcf_samples_1011)

In [ ]:
strains_1011['het_freq'] = het_freq_1011.loc[strains_1011.index]

In [ ]:
#het_freq.loc[collection.index].to_csv(f'{base_dir}script/export.csv')
#het_freq.loc[strains_info.index].to_csv(f'{base_dir}script/export.csv')

## Divergence

In [ ]:
gt_s_alt = gt_s.to_n_alt()
gt_s_hom = ((gt_s_alt>=2) & (~gt_s_het[:]))[:]

In [ ]:
div = (gt_s_hom.sum(axis=0)/sum(ref_chr_len))[:]
div = pd.Series(div, index=vcf_samples_s)

In [ ]:
# Export het levels for the Collection table
#strains_info.loc[collection.index, ['het_freq']].to_csv('/home/mathieu/postdoc_heasley/collection/het_freq.csv')
#strains_info.loc[collection.index, ['div']].to_csv('/home/mathieu/postdoc_heasley/collection/div.csv')

In [ ]:
fig, axes = plt.subplots(nrows=2, figsize=[7,7])

dat = strains_info.loc[(strains_info['species']=='cerevisiae') & (np.array([s[-4:]!='_LRH' for s in strains_info.index]))]

ax = axes[0]
sns.boxplot(x='clade', y='het_freq', legend=False, data=dat, order=clade_order, ax=ax, color='w', fliersize=0)

for clade, x in zip(clade_order, range(len(clade_order))):
    df = dat.loc[dat['clade']==clade]
    c = clade_color[clade]
    if df.shape[0] > 0:
        ax.scatter((np.random.sample(df.shape[0])-0.5)*0.5+x, df['het_freq'], color=c, s=4, lw=0, zorder=2, alpha=0.5)
        ax.text(x, 1.3*df['het_freq'].max(), f'({df.shape[0]})', size=6, rotation=90, ha='center', va='bottom')

ax.set_xticks(np.arange(len(clade_order)))
ax.set_xticklabels(clade_order, rotation=60, ha='right')
ax.set_xlabel('')
ax.set_xmargin(0.01)

ax.set_yscale('log')
ax.set_ylabel('heterozygous SNVs\nfrequency (bp$^{-1}$)')

ax = axes[1]
sns.boxplot(x='source', y='het_freq', legend=False, data=dat, order=source_color.keys(), ax=ax,
            color='w', fliersize=0)

for src, x in zip(source_order, range(len(source_order))):
    df = dat.loc[dat['source']==src]
    c = source_color[src]
    if df.shape[0] > 0:
        ax.scatter((np.random.sample(df.shape[0])-0.5)*0.5+x, df['het_freq'], color=c, s=4, lw=0, zorder=2, alpha=0.5)
        ax.text(x, 1.3*df['het_freq'].max(), f'({df.shape[0]})', size=6, rotation=90, ha='center', va='bottom')

ax.set_xticks(np.arange(len(source_order)))
ax.set_xticklabels([i.replace('_', ' ') for i in source_order], rotation=60, ha='right')
ax.set_xlabel('')
ax.set_xmargin(0.01)

ax.set_yscale('log')
ax.set_ylabel('heterozygous SNVs\nfrequency (bp$^{-1}$)')

plt.tight_layout()
sns.despine()
plt.savefig(f'{fig_path}het_levels_boxplots.png', dpi=300)
#plt.show()
plt.close()

In [ ]:
fig, ax = plt.subplots(figsize=[4,7])

dat = strains_info.loc[(strains_info['species']=='cerevisiae') & (np.array([s[-4:]!='_LRH' for s in strains_info.index]))]

sns.boxplot(y='clade', x='het_freq', legend=False, data=dat, order=clade_order, ax=ax, color='w', fliersize=0)

for clade, y in zip(clade_order, range(len(clade_order))):
    df = dat.loc[dat['clade']==clade]
    c = clade_color[clade]
    if df.shape[0] > 0:
        ax.scatter(df['het_freq'], (np.random.sample(df.shape[0])-0.5)*0.5+y, color=c, s=4, lw=0, zorder=2, alpha=0.5)
        ax.text(1.3*df['het_freq'].max(), y, f'({df.shape[0]})', size=6, ha='left', va='center')

ax.set_yticks(np.arange(len(clade_order)))
ax.set_yticklabels(clade_order, ha='right')
ax.set_ylabel('')
ax.set_ymargin(0.01)

ax.set_xscale('log')
ax.set_xlabel('heterozygous SNVs\nfrequency (bp$^{-1}$)')

plt.tight_layout()
sns.despine()
plt.savefig(f'{fig_path}het_levels_boxplots.horizontal_clade.png', dpi=300)
#plt.show()
plt.close()

In [ ]:
fig, ax = plt.subplots(figsize=[4,7])

sns.boxplot(y='source', x='het_freq', legend=False, data=dat, order=source_color.keys(), ax=ax,
            color='w', fliersize=0)

for src, y in zip(source_order, range(len(source_order))):
    df = dat.loc[dat['source']==src]
    c = source_color[src]
    if df.shape[0] > 0:
        ax.scatter(df['het_freq'], (np.random.sample(df.shape[0])-0.5)*0.5+y, color=c, s=4, lw=0, zorder=2, alpha=0.5)
        ax.text(1.3*df['het_freq'].max(), y, f'({df.shape[0]})', size=6, ha='left', va='center')

ax.set_yticks(np.arange(len(source_order)))
ax.set_yticklabels([i.replace('_', ' ') for i in source_order], ha='right')
ax.set_ylabel('')
ax.set_ymargin(0.01)

ax.set_xscale('log')
ax.set_xlabel('heterozygous SNVs\nfrequency (bp$^{-1}$)')

plt.tight_layout()
sns.despine()
plt.savefig(f'{fig_path}het_levels_boxplots.horizontal_source.png', dpi=300)
#plt.show()
plt.close()

## Heterozygosity levels comparison

In [ ]:
fig, ax = plt.subplots(figsize=[7,5])

dat = pd.concat([strains_1011.loc[(strains_1011['HO deletion']=='no') &
                 (strains_1011['Ploidy'].replace({'Xn':53}).astype(np.int64)>1), ['Clades', 'Het_Freq']].rename({'Clades':'clade'}, axis=1), 
                 strains_info.loc[(strains_info['dataset']=='yjm_collection') &
                                  (np.array([s[-4:]!='_LRH' for s in strains_info.index])), ['clade', 'het_freq', 'dataset']]])
dat['dataset'] = dat['dataset'].fillna('1011_genomes')
dat['Dataset'] = dat['dataset'].fillna('1011 genomes').replace({'yjm_collection':'YJM collection', '1011_genomes': '1011 genomes'})

mwu_het_freq_dataset = []
for c, df in dat.loc[~dat['het_freq'].isna()].groupby('clade'):
    u = df.loc[df['dataset']=='yjm_collection', 'het_freq']
    v = df.loc[df['dataset']=='1011_genomes', 'het_freq']
    if u.shape[0] > 1 and v.shape[0] > 1:
        U, p = stats.mannwhitneyu(u, v)
        mwu_het_freq_dataset.append([c, U, p])
mwu_het_freq_dataset = pd.DataFrame(mwu_het_freq_dataset, columns=['clade', 'U', 'pval'])
mwu_het_freq_dataset['pval_corr'] = multipletests(mwu_het_freq_dataset['pval'], method='fdr_bh')[1]
mwu_het_freq_dataset.set_index('clade', inplace=True)

dat['dataset'] = dat['dataset'].fillna('1011_genomes')

clade_order_sub = dat.groupby('clade').apply(lambda x: x['dataset'].unique().shape[0], include_groups=False)
clade_order_sub = [c for c in clade_order if clade_order_sub.loc[c]==2]
clade_idx_sub = dict(zip(clade_order_sub, range(len(clade_order_sub))))

sns.boxplot(x='clade', y='het_freq', hue='Dataset', data=dat, order=clade_order_sub, fliersize=0, ax=ax, zorder=1,
            hue_order=['YJM collection', '1011 genomes'], palette={'YJM collection':'w', '1011 genomes':'0.85'})
# overlay individual data points
for clade, df in dat.groupby('clade'):

    if clade in clade_order_sub:

        x = clade_idx_sub[clade]
        y = df['het_freq'].max()
        c = clade_color[clade]
    
        for ds, dx in zip(['yjm_collection', '1011_genomes'], [-0.2, 0.2]):
            df1 = df.loc[df['dataset']==ds]
            if df1.shape[0] > 0:
                ax.scatter((np.random.sample(df1.shape[0])-0.5)*0.25+dx+x, df1['het_freq'], color=c, s=4, lw=0, zorder=2, alpha=0.5)
                ax.text(x+dx, 1.3*df1['het_freq'].max(), f'({df1.shape[0]})', size=6, rotation=90, ha='center', va='bottom')
    
        if clade in mwu_het_freq_dataset.index:
            p = mwu_het_freq_dataset.loc[clade, 'pval_corr']
            ax.plot(np.array([-0.3, 0.3])+x, np.repeat(2.5*y, 2), c='k', lw=1)
            ax.text(x, 3*y, f'{plot_pval_symbol(p)}', size=8, c='k', ha='center')

ax.set_xticks(np.arange(len(clade_order_sub)))
ax.set_xticklabels(clade_order_sub, rotation=60, ha='right')
ax.set_yscale('log')

ax.legend(loc=9, bbox_to_anchor=[0.6, 1], frameon=False)

ax.set_ylim(3e-5, 5e-2)
ax.set_xlim(-0.7, 19.5)
ax.set_ylabel('heterozygous SNVs frequency (bp$^{-1}$)')
ax.set_xlabel('')


sns.despine()
plt.tight_layout()
#plt.savefig(f'{fig_path}het_freq_dataset_comparisons.png', dpi=300)
plt.show()
plt.close()

In [ ]:
# Compute heterozygosity levels by window

HET_WIN = []
wdw = 1e4
idx = 0
with ProgressBar(max_value=len(vcf_samples_s)) as bar:
    for s, i in vcf_samples_s_idx.items():
    
        df = np.concatenate([CHROM_s.reshape(-1,1), pos_bin_s.reshape(-1,1), gt_s_het[:,i].reshape(-1,1)], axis=1)
        df = pd.DataFrame(df, columns=['ctg', 'pos_bin', 'is_het'])
        df = (df.groupby(['ctg', 'pos_bin'])['is_het'].sum()/wdw).rename('het_freq').reset_index()
        df['strain'] = s
    
        HET_WIN.append(df)

        idx += 1
        bar.update(idx)

HET_WIN = pd.concat(HET_WIN).reset_index(drop=True)

HET_WIN['chrom'] = HET_WIN['ctg'].apply(lambda x: chr_ctg_to_alias[x])

In [ ]:
# Compute divergence levels by window

SNP_WIN = []
wdw = 1e4
idx = 0
with ProgressBar(max_value=len(vcf_samples_s)) as bar:
    for s, i in vcf_samples_s_idx.items():
    
        df = np.concatenate([CHROM_s.reshape(-1,1), pos_bin_s.reshape(-1,1), gt_s_hom[:,i].reshape(-1,1)], axis=1)
        df = pd.DataFrame(df, columns=['ctg', 'pos_bin', 'is_snp'])
        df = (df.groupby(['ctg', 'pos_bin'])['is_snp'].sum()/wdw).rename('snp_freq').reset_index()
        df['strain'] = s
    
        SNP_WIN.append(df)

        idx += 1
        bar.update(idx)

SNP_WIN = pd.concat(SNP_WIN).reset_index(drop=True)

SNP_WIN['chrom'] = SNP_WIN['ctg'].apply(lambda x: chr_ctg_to_alias[x])

# Het SNPS heatmaps

In [ ]:
#for subset in ['', '.high_het', '.high_12R', '.ginny']:
#for subset in ['', '.high_het', '.ginny']:
#for subset in ['', '.high_12R_ratio', '.high_12R']:
for subset in ['.high_12R']:

    fig = plt.figure(figsize=[18,18])
    gs = plt.GridSpec(ncols=19, nrows=1, width_ratios=[1.5e5]*3 + list(tig_off['len'].iloc[:-1]),
                     left=0.06, right=0.8, top=0.96, bottom=0.04, hspace=0.1)
    
    chr_ax = dict(zip(chrom_order[:-1], range(3, 19)))
    
    if subset == '':
        S = S_phylo
        fig = plt.figure(figsize=[24,24])
        
    elif subset == '.high_het':
        S = strains_info.loc[strains_info['species']=='cerevisiae'].sort_values('het_freq', ascending=False).iloc[:100]\
        .sort_values(by=['clade_idx', 'het_freq']).index
    
    elif subset == '.high_12R_ratio':
        S = HET_12R.loc[(HET_12R['het_12R_ratio']>1) &
                        (HET_12R['other']>1e-3) &
                        (HET_12R['species']=='cerevisiae'), 'strain']
    elif subset == '.high_12R':
        S = HET_12R.loc[(HET_12R['12R']>1e-3) &
                        (HET_12R['species']=='cerevisiae')].sort_values(by=['ploidy', 'rDNA'])['strain']
    
    elif subset == '.ginny':
        S = ['YJM222', 'YJM273', 'YJM308', 'YJM309', 'YJM310', 'YJM311', 'YJM312', 'YJM436', 'YJM440', 'YJM464',
             'YJM465', 'YJM466', 'YJM467', 'YJM521', 'YJM522', 'YJM523', 'YJM524', 'YJM525']
    
    for chrom, df in HET_WIN.loc[HET_WIN['chrom']!='chrmt'].groupby('chrom'):
        ax = fig.add_subplot(gs[chr_ax[chrom]])
        
        df = df.pivot_table(index='strain', columns='pos_bin', values='het_freq').astype(np.float64).loc[S]
        vmax = np.quantile(df, 0.98)
        hm = ax.imshow(df.loc[S], cmap='viridis', aspect='auto', interpolation='nearest', vmin=0, vmax=vmax, zorder=1)

        chrom_len = tig_off.loc[chrom, 'len']
        centromere_position = (centromeres[chr_alias_to_ctg[chrom]]/chrom_len)*df.shape[1]
        ax.axvline(centromere_position, c='red', lw=0.5, zorder=2)
        if chrom == 'chrXII':
            ax.axvline(470522/chrom_len*df.shape[1], c='cyan', lw=1, zorder=2) 
        
        ax.set_title(chrom[3:])

        for s in ['top', 'bottom', 'left', 'right']:
            ax.spines[s].set_visible(False)
        ax.set_yticks([])
        x_ticks = np.arange(0, df.shape[1], 20)
        ax.set_xticks(x_ticks)
        ax.set_xticklabels([f'{i:.0f}' for i in x_ticks*1e4*1e-3], rotation=90, size=6)
    
    # Annotate clades
    
    ax = fig.add_subplot(gs[0])

    cmap = colors.ListedColormap(list(clade_color.values()))
    bounds = np.arange(len(clade_order)+1)
    norm = colors.BoundaryNorm(bounds, cmap.N)
    
    dat = strains_info.loc[S, 'clade_idx'].values.reshape(-1,1)
    ax.imshow(dat, cmap=cmap, norm=norm, aspect='auto', interpolation='nearest')
    
    for s in ['top', 'bottom', 'left', 'right']:
        ax.spines[s].set_visible(False)
    
    ax.set_yticks(np.arange(len(S)))
    if len(S) > 100:
        ax.set_yticklabels(S, size=4)
    else:
        ax.set_yticklabels(S)
    ax.set_xticks([])
    ax.set_title('Clade', rotation=90)

    # Annotate sources
    
    ax = fig.add_subplot(gs[1])

    cmap = colors.ListedColormap(list(source_color.values()))
    bounds = np.arange(len(source_order)+1)
    norm = colors.BoundaryNorm(bounds, cmap.N)
    
    dat = strains_info.loc[S, 'source'].apply(lambda x: source_idx[x]).values.reshape(-1,1)
    ax.imshow(dat, cmap=cmap, norm=norm, aspect='auto', interpolation='nearest')
    
    for s in ['top', 'bottom', 'left', 'right']:
        ax.spines[s].set_visible(False)
    
    ax.set_yticks([])
    ax.set_xticks([])
    ax.set_title('Source', rotation=90)

    # Annotate ploidy
    
    ax = fig.add_subplot(gs[2])

    bounds = np.arange(len(source_order)+1)
    norm = colors.BoundaryNorm(bounds, cmap.N)
    
    dat = strains_info.loc[S, 'ploidy'].values.reshape(-1,1)
    ax.imshow(dat, cmap='Greens', vmin=1, vmax=4, aspect='auto', interpolation='nearest')
    
    for s in ['top', 'bottom', 'left', 'right']:
        ax.spines[s].set_visible(False)
    
    ax.set_yticks([])
    ax.set_xticks([])
    ax.set_title('Ploidy', rotation=90)

    # Add colorbar

    ax_cbar = fig.add_axes([0.83, 0.5, 0.14, 0.02])
    cb = plt.colorbar(hm, cax=ax_cbar, orientation='horizontal')
    cb.outline.set_visible(False)
    cb_x_ticks = np.linspace(0, vmax, 5)
    ax_cbar.set_xticks(cb_x_ticks)
    ax_cbar.set_xticklabels([f'{x:.3f}' for x in cb_x_ticks*100])
    ax_cbar.text(0.5, 1.05, 'heterozygosity rate in\n10 kb windows (%)', 
                 ha='center', va='bottom', transform=ax_cbar.transAxes)
    
    
    ax_legend = fig.add_axes([0.8, 0.5, 0.2, 0.5])
    ax_legend.axis('off')
    legend_elms = [Line2D([0], [0], c='w', marker='s', ms=12, mfc=c, label=l) for (l,c) in clade_color.items()]
    ax_legend.legend(handles=legend_elms, ncol=1, loc=6, bbox_to_anchor=[0.08, 0.5], frameon=False, title='Clade')

    ax_legend = fig.add_axes([0.8, 0, 0.2, 0.5])
    ax_legend.axis('off')
    legend_elms = [Line2D([0], [0], c='w', marker='s', ms=12, mfc=c, label=l) for (l,c) in source_color.items()]
    ax_legend.legend(handles=legend_elms, ncol=1, loc=6, bbox_to_anchor=[0.08, 0.5], frameon=False, title='Source')
    
    
    plt.savefig(f'{fig_path}hm_het_freq{subset}.png', dpi=300)
    #plt.show()
    plt.close()

# Competitive mapping on combined reference genome

In [ ]:
samtools_stats = pd.read_csv(f'{base_dir}tables/samtools_stats.tsv', sep='\t', index_col=0)

In [ ]:
combined_reference = [seq for seq in SeqIO.parse(f'{base_dir}data/ref/combined.fasta', 'fasta')]
combined_ref_chrlen = pd.Series({seq.id:len(seq) for seq in combined_reference})

In [ ]:
species_order = ['cerevisiae', 'paradoxus', 'mikatae', 'jurei', 'kudriavzevii', 'arboricola', 'eubayanus', 'uvarum']
species_order_idx = dict(zip(species_order, range(8)))

tig_prefix_species = pd.read_csv(f'{base_dir}script/ctg_prefix_species.tsv', sep='\t').set_index('contig')
tig_prefix_species['species_idx'] = tig_prefix_species['species'].apply(lambda x: species_order_idx[x])
tig_prefix_species['species_chrom'] = tig_prefix_species.apply(lambda x: f'{x["species"]}.{x["chr"]}', axis=1)
tig_prefix_species = tig_prefix_species.sort_values(by=['species_idx', 'chr'])
tig_prefix_species['idx'] = np.arange(tig_prefix_species.shape[0])
species_chrom_order = list(tig_prefix_species['species_chrom'].values)



strains_info_combined = pd.read_csv(f'{base_dir}script/strains_info_combined.csv').set_index('strain')
S_combined = strains_info_combined.loc[strains_info_combined['ref']=='combined'].index

In [ ]:
DEPTH_COMBINED = []

for s in S_combined:
    depth = pd.read_csv(f'{base_dir}data/depth/{s}.depth_nuc_bin.tab.gz', sep='\t', header=None)
    depth['strain'] = s
    depth['bases_estimate'] = depth[2]*50000
    DEPTH_COMBINED.append(depth)
DEPTH_COMBINED = pd.concat(DEPTH_COMBINED).reset_index(drop=True)
DEPTH_COMBINED['species'] = tig_prefix_species.loc[DEPTH_COMBINED[0], 'species'].values
DEPTH_COMBINED['chrom'] = tig_prefix_species.loc[DEPTH_COMBINED[0], 'chr'].values
DEPTH_COMBINED['bases_mapped'] = samtools_stats.loc[DEPTH_COMBINED['strain'], 'bases mapped'].values
DEPTH_COMBINED['bases_norm'] = DEPTH_COMBINED['bases_estimate']/DEPTH_COMBINED['bases_mapped']

In [ ]:
strain_cb_median = DEPTH_COMBINED.groupby('strain')['bases_estimate'].apply(lambda x: np.nanmedian(np.where(x==0, np.nan, x)))

In [ ]:
# Summary table by species
dp_cb = DEPTH_COMBINED.groupby(['strain', 'species'])['bases_norm'].sum().rename('sum_dp').reset_index()
# summary table by species-chromosome
dp_cb_ctg = DEPTH_COMBINED.groupby(['strain', 'species', 0, 'chrom'])['bases_estimate'].sum().rename('sum_dp').reset_index()
dp_cb_ctg['sum_dp_norm'] = dp_cb_ctg['sum_dp']/combined_ref_chrlen.loc[dp_cb_ctg[0]].values
dp_cb_ctg['species_chrom'] = dp_cb_ctg.apply(lambda x: f'{x["species"]}.{x["chrom"]}', axis=1)
dp_cb_ctg['species_chrom_idx'] = tig_prefix_species.set_index('species_chrom').loc[dp_cb_ctg['species_chrom'], 'idx'].values

In [ ]:
combined_ref_chrlen[dp_cb_ctg.set_index('species_chrom').loc[[f'eubayanus.{i}' for i in range(17,25)], 0].unique()]

In [ ]:
fig, ax = plt.subplots(figsize=[18,5])

dat = dp_cb_ctg.pivot_table(index='strain', columns='species_chrom', values='sum_dp_norm')
sns.heatmap(dat.loc[S_combined, species_chrom_order].replace({0:np.nan}), cmap='viridis', vmin=0, vmax=200, ax=ax)

ax.set_yticklabels([s.split('_')[0] for s in S_combined])

ax.set_xticks(np.arange(dat.shape[1])+0.5)
ax.set_xticklabels(species_chrom_order, rotation=90, size=6, ha='center')

plt.tight_layout()
plt.savefig(f'{fig_path}hm_competitive_mapping_chr.png', dpi=300)
plt.show()
plt.close()

In [ ]:
fig, ax = plt.subplots(figsize=[6,6])

dat = dp_cb.pivot_table(index='strain', columns='species', values='sum_dp') * 100
sns.heatmap(dat.loc[S_combined, species_order], cmap='viridis', ax=ax, vmin=0, vmax=100,
            annot=True, annot_kws={'size':8}, fmt='.1f', cbar_kws={'label':'% mapped reads'})
ax.set_yticklabels([s.split('_')[0] for s in S_combined])

plt.tight_layout()
plt.savefig(f'{fig_path}hm_competitive_mapping.png', dpi=300)
plt.show()
plt.close()

# Depth

In [ ]:
# Import depth
DEPTH_NUC = []
idx = 0

with ProgressBar(max_value=strains_info.shape[0]) as bar:
    
    for s in strains_info.index:
        depth = pd.read_csv(f'{base_dir}data/depth/{s}.depth_nuc_bin.tab.gz', header=None, sep='\t')
        depth.columns = ['CHROM', 'POS', 'DP']
        depth['dp_rel'] = depth['DP']/depth.loc[depth['CHROM'].isin(nuclear_chromosomes), 'DP'].median()
        
        depth['strain'] = s

        DEPTH_NUC.append(depth)

        idx += 1
        bar.update(idx)

DEPTH_NUC = pd.concat(DEPTH_NUC).reset_index(drop=True)
DEPTH_NUC['species'] = strains_info.loc[DEPTH_NUC['strain'], 'species'].values

# Repeats CN

In [ ]:
DP_REP = []
for s, df in DEPTH_NUC.groupby('strain'):
    dp_nuc = df.loc[df['CHROM'].isin(nuclear_chromosomes), 'DP'].median()
    df = df.loc[df['CHROM'].isin(repeats)]
    added_repeat = {r:False for r in repeats}
    for r, df1 in df.groupby('CHROM'):
        DP_REP.append([s, r, df1['DP'].median(), dp_nuc])
        added_repeat[r] = True
    for r,a in added_repeat.items():
        if a == False:
            DP_REP.append([s, r, 0, dp_nuc])
DP_REP = pd.DataFrame(DP_REP, columns=['strain', 'repeat', 'depth', 'depth_nuc'])
DP_REP['cn'] = DP_REP['depth']/DP_REP['depth_nuc']
DP_REP['clade'] = strains_info.loc[DP_REP['strain'], 'clade'].values
DP_REP['source'] = strains_info.loc[DP_REP['strain'], 'source'].values
DP_REP['source_simplified'] = DP_REP['source'].apply(lambda x: source_simplified_order[source_simplified_dict[x]])
DP_REP['species'] = strains_info.loc[DP_REP['strain'], 'species'].values
DP_REP['ploidy'] = strains_info.loc[DP_REP['strain'], 'ploidy'].values
# add z-score transformed data
for r, df in DP_REP.groupby('repeat'):
    z = (df['cn']-df['cn'].mean())/df['cn'].std()
    DP_REP.loc[df.index, 'cn_z_score'] = z

In [ ]:
strains_info_strope = pd.read_csv(f'{base_dir}wine_subclade_strope/strains_info.csv').set_index('strain', drop=False)
wine_subclade_spores_parents = dict(zip(strains_info_strope['parent'], strains_info_strope['strain']))
DP_REP_STROPE = []

for s in strains_info_strope.index:
    depth = pd.read_csv(f'{base_dir}data/depth/{s}.depth_nuc_bin.tab.gz', header=None, sep='\t')
    depth.columns = ['CHROM', 'POS', 'DP']
    dp_nuc = depth.loc[depth['CHROM'].isin(nuclear_chromosomes), 'DP'].median()
    df = depth.loc[depth['CHROM'].isin(repeats)]
    added_repeat = {r:False for r in repeats}
    for r, df1 in df.groupby('CHROM'):
        DP_REP_STROPE.append([s, r, df1['DP'].median(), dp_nuc])
        added_repeat[r] = True
    for r,a in added_repeat.items():
        if a == False:
            DP_REP_STROPE.append([s, r, 0, dp_nuc])

DP_REP_STROPE = pd.DataFrame(DP_REP_STROPE, columns=['strain', 'repeat', 'depth', 'depth_nuc'])
DP_REP_STROPE['cn'] = DP_REP_STROPE['depth']/DP_REP_STROPE['depth_nuc']

## Fig 1A-C

In [ ]:
fig = plt.figure(figsize=[11,4])
gs = plt.GridSpec(ncols=3, nrows=3, hspace=0.6, wspace=0.3, height_ratios=[2,2,1], width_ratios=[3,6,1], 
                 left=0.07, right=0.98, top=0.86, bottom=0.33)

f_strains_dup = ~strains_info['is_duplicate']
f_strains_yjm = strains_info['dataset']!='1011_genomes'
f_strains_cer = strains_info['species']=='cerevisiae'

# copy number standard deviation
ax = fig.add_subplot(gs[0,0])

S = strains_info.loc[f_strains_cer & f_strains_dup].index

repeats_order_std = DP_REP.loc[DP_REP['repeat']!='ref|NC_001224|']\
    .pivot_table(index='strain', columns='repeat', values='cn').loc[S]\
    .std().sort_values()

X = np.arange(repeats_order_std.shape[0])
ax.plot(X, repeats_order_std, c='k', marker='o', clip_on=False)

ax.set_xticks(X, labels=[])
ax.set_ylabel('CN SD')

# copy number boxplots
ax = fig.add_subplot(gs[1:,0])

dat = DP_REP.loc[DP_REP['strain'].isin(S)]

sns.boxplot(x='repeat', y='cn', order=repeats_order_std.index,
             data=dat, color='w', linecolor='k', ax=ax)

ax.set_xticks(X, labels=[repeats_alias[t.get_text()] for t in ax.get_xticklabels()],
              rotation=90)

ax.set_xlabel('')
ax.set_ylabel('CN')

# Y' copy numbers per clade
ax = fig.add_subplot(gs[:2,1])

dat = DP_REP.loc[(DP_REP['strain'].isin(S)) & (DP_REP['repeat']=='Y_prime_element.cons_trim')]
sns.boxplot(x='clade', y='cn', orient='v', order=clade_order,
            hue='clade', legend=False, palette=clade_color, data=dat)
ax.set_xlabel('')
ax.set_xticks(range(len(clade_order)))
ax.set_xticklabels(clade_order, rotation=90)
ax.set_ylabel('Y\' element CN')

sns.despine()

# Y' copy numbers per sampling source
ax = fig.add_subplot(gs[:,2])

dat = DP_REP.loc[(DP_REP['strain'].isin(S)) & (DP_REP['repeat']=='Y_prime_element.cons_trim')]
sns.boxplot(x='source_simplified', y='cn', orient='v', order=source_simplified_order,
            hue='source_simplified', legend=False, palette=source_simplified_color, data=dat)
ax.set_xlabel('')
ax.set_xticks(range(len(source_simplified_order)))
ax.set_xticklabels(source_simplified_order, rotation=90)

ax.set_ylabel('Y\' element CN')

sns.despine()

fig.text(0.02, 0.91, 'A', size=24, fontweight='bold')
fig.text(0.32, 0.91, 'B', size=24, fontweight='bold')
fig.text(0.84, 0.91, 'C', size=24, fontweight='bold')

#for ext in ['png', 'svg']:
#    plt.savefig(f'{fig_path}Fig1A-C.{ext}', dpi=300)
    
plt.show()
plt.close()

In [ ]:
## FE test extreme CN vs clinical
S = strains_info.loc[f_strains_cer & f_strains_dup & f_strains_yjm].index
dat = DP_REP.loc[(DP_REP['strain'].isin(S)) & (DP_REP['repeat']=='Y_prime_element.cons_trim')].copy()
dat['clinical'] = np.where(dat['source']=='clinical', 1, 0)
dat['extreme'] = np.where(dat['cn']>60, 1, 0)

ct = dat.value_counts(['clinical', 'extreme']).reset_index().pivot_table(index='clinical', columns='extreme')
fe = stats.fisher_exact(ct)
plt.text(0.5, 0.5, mh.plot_pval_text(fe.pvalue), ha='center', va='center', size=160)
plt.gca().axis('off')

# Chrom CN

In [ ]:
DP_CHROM = []
for s, df in DEPTH_NUC.groupby('strain'):
    dp_nuc = df.loc[df['CHROM'].isin(nuclear_chromosomes), 'DP'].median()
    df = df.loc[df['CHROM'].isin(nuclear_chromosomes)]
    added_chrom = {c:False for c in nuclear_chromosomes}
    for c, df1 in df.groupby('CHROM'):
        DP_CHROM.append([s, c, df1['DP'].median(), dp_nuc])
        added_chrom[c] = True
    for c,a in added_chrom.items():
        if a == False:
            DP_CHROM.append([s, c, 0, dp_nuc])
DP_CHROM = pd.DataFrame(DP_CHROM, columns=['strain', 'chrom', 'depth', 'depth_nuc'])
DP_CHROM['cn'] = DP_CHROM['depth']/DP_CHROM['depth_nuc']
DP_CHROM['clade'] = strains_info.loc[DP_CHROM['strain'], 'clade'].values
DP_CHROM['source'] = strains_info.loc[DP_CHROM['strain'], 'source'].values
DP_CHROM['species'] = strains_info.loc[DP_CHROM['strain'], 'species'].values
DP_CHROM['ploidy'] = strains_info.loc[DP_CHROM['strain'], 'ploidy'].values
# add z-score transformed data
for c, df in DP_CHROM.groupby('chrom'):
    z = (df['cn']-df['cn'].mean())/df['cn'].std()
    DP_CHROM.loc[df.index, 'cn_z_score'] = z
DP_CHROM['chrom_cn'] = DP_CHROM['cn']*DP_CHROM['ploidy']

In [ ]:
# export for the collection table
#DP_CHROM.pivot_table(index='strain', columns='repeat', values='cn', aggfunc=lambda x: x)\
#.loc[collection['ID'], repeats_order].rename(columns=repeats_alias).to_csv('/home/mathieu/postdoc_heasley/collection/repeats_CN.tsv', sep='\t')

In [ ]:
fig, ax = plt.subplots(figsize=[9,7])

dat = DP_REP.loc[DP_REP['species']=='cerevisiae'].pivot_table(index='clade', columns='repeat', values='cn_z_score').fillna(0)
# Z-score normalization
#dat = (dat-dat.mean())/dat.std()

sns.heatmap(dat.loc[clade_order, repeats_order].T, ax=ax, vmin=np.quantile(dat, 0.02), vmax=np.quantile(dat, 0.98),
           cmap='viridis', zorder=1, cbar_kws={'label':'copy number Z-score'})
ax.set_yticks(np.arange(len(repeats))+0.5)
ax.set_yticklabels([repeats_alias[r] for r in repeats_order])

for y in [1,2,4]:
    ax.axhline(y, c='w', zorder=2)

plt.tight_layout()

#plt.savefig(f'{fig_path}hm_cn_repeats_by_clade.png', dpi=300)
plt.show()
plt.close()

In [ ]:
for r, df in DP_REP.loc[(DP_REP['species']=='cerevisiae')].groupby('repeat'):
    
    #if r == 'Y_prime_element.cons_trim':
        
    R = repeats_alias[r]

    fig, axes = plt.subplots(nrows=2, figsize=[7,5])

    # By clade

    ax = axes[0]
    x_order = df.groupby('clade')['cn'].mean().sort_values().index
    
    sns.boxplot(x='clade', y='cn', data=df, order=x_order, color='w', fliersize=0, ax=ax)

    for clade, x in zip(x_order, range(len(x_order))):
        df1 = df.loc[df['clade']==clade]
        c = clade_color[clade]
        if df1.shape[0] > 0:
            ax.scatter((np.random.sample(df1.shape[0])-0.5)*0.5+x, df1['cn'], color=c, s=4, lw=0, zorder=2, alpha=0.5)

    ax.set_title(R)
    ax.set_ylabel('Copy number')
    ax.set_xlabel('')
    
    ax.set_xticks(np.arange(len(x_order)))
    ax.set_xticklabels(x_order, rotation=60, ha='right', size=7)
    
    # By source

    ax = axes[1]
    x_order = df.groupby('source')['cn'].mean().sort_values().index  
    
    sns.boxplot(x='source', y='cn', data=df, order=x_order, color='w', fliersize=0, ax=ax)
    for src, x in zip(x_order, range(len(x_order))):
        df1 = df.loc[df['source']==src]
        c = source_color[src]
        if df1.shape[0] > 0:
            ax.scatter((np.random.sample(df1.shape[0])-0.5)*0.5+x, df1['cn'], color=c, s=4, lw=0, zorder=2, alpha=0.5)
    
    ax.set_title(R)
    ax.set_ylabel('Copy number')
    ax.set_xlabel('')
    
    ax.set_xticks(np.arange(len(x_order)))
    ax.set_xticklabels([i.replace('_', ' ') for i in x_order], rotation=60, ha='right', size=7)

    sns.despine()
    plt.tight_layout()
    plt.savefig(f'{fig_path}cn_{R}_by_clade_source.png', dpi=300)   
    #plt.show()
    plt.close()

In [ ]:
r = 'Y_prime_element.cons_trim'
df = DP_REP.loc[(DP_REP['species']=='cerevisiae') & (DP_REP['repeat']==r)]

fig, ax = plt.subplots(figsize=[4,7])

# By clade

y_order = df.groupby('clade')['cn'].mean().sort_values(ascending=False).index

sns.boxplot(y='clade', x='cn', data=df, order=y_order, color='w', fliersize=0, ax=ax)

for clade, y in zip(y_order, range(len(y_order))):
    df1 = df.loc[df['clade']==clade]
    c = clade_color[clade]
    if df1.shape[0] > 0:
        ax.scatter(df1['cn'], (np.random.sample(df1.shape[0])-0.5)*0.5+y, color=c, s=4, lw=0, zorder=2, alpha=0.5)

ax.set_xlabel('Y\' element copy number')
ax.set_ylabel('')

ax.set_yticks(np.arange(len(y_order)))
ax.set_yticklabels(y_order, ha='right')

sns.despine()
plt.tight_layout()
plt.savefig(f'{fig_path}cn_{repeats_alias[r]}_by_clade.vertical.png', dpi=300)
#plt.show()
plt.close()

In [ ]:
# By source

fig, ax = plt.subplots(figsize=[4,7])

y_order = df.groupby('source')['cn'].mean().sort_values(ascending=False).index  

sns.boxplot(y='source', x='cn', data=df, order=y_order, color='w', fliersize=0, ax=ax)
for src, y in zip(y_order, range(len(y_order))):
    df1 = df.loc[df['source']==src]
    c = source_color[src]
    if df1.shape[0] > 0:
        ax.scatter(df1['cn'], (np.random.sample(df1.shape[0])-0.5)*0.5+y, color=c, s=4, lw=0, zorder=2, alpha=0.5)

ax.set_xlabel('Y\' element copy number')
ax.set_ylabel('')

ax.set_yticks(np.arange(len(y_order)))
ax.set_yticklabels([i.replace('_', ' ') for i in y_order], ha='right')

sns.despine()
plt.tight_layout()
plt.savefig(f'{fig_path}cn_{repeats_alias[r]}_by_source.vertical.png', dpi=300)
#plt.show()
plt.close()

In [ ]:
clades_with_clinical = strains_info.loc[strains_info['source']=='clinical'].value_counts('clade')
clades_with_clinical = list(clades_with_clinical.loc[clades_with_clinical>=3].index)
clades_with_clinical_idx = dict(zip(clades_with_clinical, range(len(clades_with_clinical))))

In [ ]:
dat = DP_REP.loc[DP_REP['clade'].isin(clades_with_clinical)].copy()
dat['clinical'] = dat['source'] == 'clinical'

for r, df in dat.groupby('repeat'):
    if r == 'Y_prime_element.cons_trim':
        fig, ax = plt.subplots(figsize=[6,6])
    
        mwu_cn_clinical = []
        for clade, df1 in df.groupby('clade'):
            u = df1.loc[df1['clinical']==True, 'cn']
            v = df1.loc[df1['clinical']==False, 'cn']
            #if u.shape[0] > 1 and v.shape[0] > 1:
            U, p = stats.mannwhitneyu(u, v)
            mwu_cn_clinical.append([clade, U, p])
        mwu_cn_clinical = pd.DataFrame(mwu_cn_clinical, columns=['clade', 'U', 'pval'])
        mwu_cn_clinical['pval_corr'] = multipletests(mwu_cn_clinical['pval'], method='fdr_bh')[1]
        mwu_cn_clinical.set_index('clade', inplace=True)
    
        sns.boxplot(x='clade', y='cn', hue='clinical', data=df, order=clades_with_clinical, fliersize=0, ax=ax, zorder=1,
                    hue_order=[False, True], palette={False:'w', True:'0.85'})
        
        # overlay individual data points
        for clade, df1 in df.groupby('clade'):
        
            x = clades_with_clinical_idx[clade]
            y = df1['cn'].max()
            dy = 0.15*y
            c = clade_color[clade]
        
            for cl, dx in zip([False, True], [-0.2, 0.2]):
                df2 = df1.loc[df['clinical']==cl]
                if df2.shape[0] > 0:
                    ax.scatter((np.random.sample(df2.shape[0])-0.5)*0.25+dx+x, df2['cn'], color=c, s=4, lw=0, zorder=2, alpha=0.5)
                    ax.text(x+dx, df2['cn'].max(), f'({df2.shape[0]})', size=6, rotation=90, ha='center', va='bottom')
        
            if clade in mwu_cn_clinical.index:
                p = mwu_cn_clinical.loc[clade, 'pval_corr']
                ax.plot(np.array([-0.3, 0.3])+x, np.repeat(y+dy, 2), c='k', lw=1)
                ax.text(x, y+dy, f'{plot_pval_symbol(p)}', size=8, c='k', ha='center')
    
    
        ax.set_xticks(range(len(clades_with_clinical)))
        ax.set_xticklabels(clades_with_clinical, rotation=60, ha='right')
    
        ax.set_title(repeats_alias[r])
        
        plt.show()
        plt.close()

# Admixture

In [ ]:
K_values = range(3, 41)
K_cv = {}
ADM_Q = {}
for k in K_values:
    admQ = pd.read_csv(f'{base_dir}admixture/mpileup_all.add_tags.fil.corr.renamed.pruneLD.{k}.Q', sep=' ', header=None)
    ADM_Q[k] = admQ
    log_path = f'{base_dir}admixture/{k}.out.log'
    with open(log_path) as handle:
        for line in handle.readlines():
            m = re.match(r'CV error \(K=\d{1,2}\): ([\d\.]+)', line)
            if m:
                K_cv[k] = np.float32(m.group(1))

K_cv = pd.Series(K_cv)

In [ ]:
fig, ax = plt.subplots()
ax.plot(K_cv.index, K_cv, c='k')
ax.set_ylabel('CV error')
ax.set_xlabel('k')


plt.tight_layout()
sns.despine()

for ext in ['png', 'svg']:
    plt.savefig(f'{fig_path}FigS1.{ext}', dpi=300)
#plt.show()
plt.close()

In [ ]:
k = 29


admQ = ADM_Q[k]
admQ_cumul = np.cumsum(admQ, axis=1)

ADM_MAJ = []
admQ = ADM_Q[k]
for i in admQ.index:
    maj_anc = admQ.loc[i].sort_values(ascending=False).index[0]
    maj_Q = admQ.loc[i, maj_anc]
    s = S_adm[i]
    ADM_MAJ.append([s, maj_anc, maj_Q, colorcet.glasbey_category10[maj_anc]])
ADM_MAJ = pd.DataFrame(ADM_MAJ, columns=['strain','maj_anc','anc_Q','color'])
ADM_MAJ['mosaic'] = ADM_MAJ['anc_Q'] < 0.6
ADM_MAJ['clade'] = strains_info.loc[ADM_MAJ['strain'], 'clade'].values
ADM_MAJ = ADM_MAJ.set_index('strain')

In [ ]:
S_adm = list(strains_info.loc[strains_info['species']=='cerevisiae'].index)
S_adm_sorted = list(strains_info.loc[S_adm].sort_values(by=['clade','strain']).index)
S_adm_idx = dict(zip(S_adm, range(422)))
S_adm_sorted_idx = [S_adm_idx[s] for s in S_adm_sorted]

In [ ]:
fig, ax = plt.subplots(figsize=[9,9])

S = ADM_MAJ.sort_values(by=['maj_anc','anc_Q'], ascending=[True,False]).index
S_idx = [S_adm_idx[s] for s in S]

for s, (x,y) in zip(S, [(i,j) for (i,j) in itertools.product(range(21), range(21))]):
    mfc, mosaic, clade = ADM_MAJ.loc[s, ['color', 'mosaic', 'clade']]
    tc = clade_color[clade]
    if mosaic:
        mec = 'red'
    else:
        mec = 'k'

    ax.scatter(x, y, color=mfc, edgecolors=mec, s=150)
    ax.text(x, y, s, color=tc, size=5, ha='center', va='center')

ax.invert_yaxis()

plt.tight_layout()
#plt.savefig(f'{fig_path}admixture/k36_mosaic60.png', dpi=500)
plt.show()
plt.close()

# Phylo

In [ ]:
gff_genes = gff.loc[gff[2]=='gene'].copy()
gff_genes = pd.concat([gff_genes.iloc[:, :8], gff_genes[8].apply(lambda x: pd.Series(dict([i.split('=') for i in x.split(';')])))], axis=1)
gff_genes.index = gff_genes['ID'].values

gff_genes['left_bound'] = np.int64(gff_genes[3]*1e-4)*10000+5000
gff_genes['right_bound'] = np.int64(gff_genes[4]*1e-4)*10000+5000

In [ ]:
DEPTH_NUC_reidx = DEPTH_NUC.set_index(['strain', 'CHROM', 'POS']).sort_index()

DP_GENES = []

idx = 0
with ProgressBar(max_value=gff_genes.shape[0]) as bar:
    
    for g in gff_genes.loc[gff_genes[0].isin(nuclear_chromosomes)].index:
        chrom, lb, rb = gff_genes.loc[g, [0, 'left_bound', 'right_bound']]
        
        for s in strains_info.index:
            dp = DEPTH_NUC_reidx.loc[(s, chrom)].loc[lb:rb, 'dp_rel'].mean()
            
            DP_GENES.append([s, g, chrom, lb, rb, dp])

        idx += 1
        bar.update(idx)

DP_GENES = pd.DataFrame(DP_GENES, columns=['strain', 'gene', 'chrom', 'left_bound', 'right_bound', 'dp_rel'])

In [ ]:
DP_GENES_VAR = pd.concat([DP_GENES.groupby('gene')['dp_rel'].mean().rename('dp_mean'),
                          DP_GENES.groupby('gene')['dp_rel'].std().rename('dp_std')], axis=1)

In [ ]:
fig, axes = plt.subplots(ncols=2, nrows=2, gridspec_kw=({'height_ratios':[1,2], 'width_ratios':[2,1]}))

ax = axes[1,1]
ax.plot(np.linspace(0, 100, DP_GENES_VAR['dp_std'].shape[0]), DP_GENES_VAR['dp_std'].sort_values(), color='k')
q50 = np.quantile(DP_GENES_VAR['dp_std'], 0.5)
ax.axhline(q50, lw=0.5, color='red')
ax.text(0.5, 0.5, f'{q50:.3f}', color='red', ha='center', va='center', transform=ax.transAxes)

ax = axes[0,0]
ax.plot(DP_GENES_VAR['dp_mean'].sort_values(), np.linspace(0, 100, DP_GENES_VAR['dp_mean'].shape[0]), color='k')
q25 = np.quantile(np.abs(DP_GENES_VAR['dp_mean']-1), 0.25)
ax.axvline(1-q25, lw=0.5, color='red')
ax.axvline(1+q25, lw=0.5, color='red')
ax.text(0.5, 0.5, f'[{1-q25:.3f}-{1+q25:.3f}]', color='red', ha='center', va='center', transform=ax.transAxes)

ax = axes[1,0]
ax.scatter(DP_GENES_VAR['dp_mean'], DP_GENES_VAR['dp_std'], color='k', s=1, alpha=0.1)
ax.set_xlabel('mean')
ax.set_ylabel('std')

ax.axhline(q50, lw=0.5, color='red')
ax.axvline(1-q25, lw=0.5, color='red')
ax.axvline(1+q25, lw=0.5, color='red')

plt.show()
plt.close()

In [ ]:
genes_selected = DP_GENES_VAR.loc[(DP_GENES_VAR['dp_mean']>=0.976)
& (DP_GENES_VAR['dp_mean']<=1.024)
& (DP_GENES_VAR['dp_std']<=0.112)].index
np.random.seed(42)
n = 500
genes_selected_random = np.random.choice(genes_selected, n)

# Export in GFF format
gff_random_sel = gff_genes.loc[genes_selected_random, [0,1,2,3,4,5,6,7,'ID']]
#gff_random_sel.to_csv(f'{base_dir}phylo/orf_coding.random{n}.gff', sep='\t', header=None, index=None)

## Concatenate sequences in order

In [ ]:
n = 500

fasta_ids_order = gff_random_sel.sort_values(by=[0,3]).apply(lambda x: f'{x[0]}_{x[3]}-{x[4]}:{x[6]}', axis=1)

fasta = {}

for s in strains_info.loc[strains_info['species']=='cerevisiae'].index:

    fasta[s] = Seq.Seq('')
    
    with open(f'{base_dir}phylo/{s}.orf_coding.random{n}.fasta') as handle:
        fasta_dict = {seq.id:seq for seq in SeqIO.parse(handle, 'fasta')}
        
    for i in fasta_ids_order:
        if i in fasta_dict.keys():
            fasta[s] += fasta_dict[i].seq
        else:
            print(f'{i} {s} absent!!')
    
for s, seq in fasta.items():
    sr = SeqRecord.SeqRecord(seq=seq, id=s, description='')
    with open(f'{base_dir}phylo/{s}.orf_coding.random{n}.concat.fasta', 'w') as handle:
        SeqIO.write(sr, handle, 'fasta')

In [ ]:
def color_internal_branch(conf, zero=0.3):
    conf = np.int64(conf)/100
    color = (1-conf)*zero
    return str(color)

## Fig S1A

In [ ]:
tree_raxml_path = f'{base_dir}raxml/GTGTR4_outgroup/orf_coding.random500.concat.fasta.raxml.support.icytree_reordered_unquoted'

# get the raw tree to add bootstrap values to the reordered one
tree_raxml_raw_path = f'{base_dir}raxml/GTGTR4_outgroup/orf_coding.random500.concat.fasta.raxml.support'
tree_raxml_raw =  Phylo.read(tree_raxml_raw_path, 'newick')

## Init figure

fig = plt.figure(figsize=[10, 9])
gs = plt.GridSpec(ncols=7, nrows=1, width_ratios=[20, 1, 1, 3, 1, 1, 1], wspace=0.4,
                 left=0.03, top=0.93, right=0.88, bottom=0.03)

ax = fig.add_subplot(gs[0])

tree_raxml = TreeViz(tree_raxml_path, leaf_label_size=0)
t = tree_raxml.tree

tree_raxml.set_node_line_props('N_1', lw=0.7, color='k', descendent=True)

for s in tree_raxml.leaf_labels:
    clade = strains_info.loc[s, 'clade']
    mfc, mec = clades_unique.loc[clade, ['mfc','mec']].values
    tip_marker = tree_raxml.marker(s, marker='o', linewidths=0.5, alpha=1, size=2.5, color=mfc, ec=mec)

for s, (x,y) in tree_raxml.name2xy_right.items():
    if s in tree_raxml.leaf_labels:
        clade = strains_info.loc[s, 'clade']
        mfc = clades_unique.loc[clade, 'mfc']
        fa = FancyArrow(x, y, 0.056-x, 0, fc=mfc, width=1, head_length=0, head_width=0, lw=0, alpha=0.2, 
                        zorder=-1, clip_on=False)
        ax.add_patch(fa)

#tree_raxml.show_scale_bar(loc='lower left', text_size=10)
tree_raxml.plotfig(ax=ax)
scale_bar = np.array([0, 0.005])+0.0005
ax.plot(scale_bar, [32,32], c='k')
ax.text(np.mean(scale_bar), 29, '0.005', ha='center', va='top')

S = [c.name for c in tree_raxml.tree.get_terminals()]
Y = np.arange(len(S))

ax.set_xlim(0, 0.032)
ax.set_ylim(Y[[0, -1]]+np.array([0.5, 1.5]))

legend_elms = [Line2D([0], [0], color='w', marker='o', ms=8, label=c, mfc=mfc, mec=mec) for c, (mfc, mec) in clades_unique[['mfc','mec']].iterrows()]
ax.legend(handles=legend_elms, title='Clade', bbox_to_anchor=[0, 1], loc=2, frameon=False, fontsize=10)

# Add annotation data

## Isolation source

ax = fig.add_subplot(gs[1])

dat = strains_info.loc[S, 'source'].apply(lambda x: source_simplified_dict[x]).values.reshape(-1,1)
ax.imshow(dat, cmap=cmap_source_simplified_color, interpolation='nearest', aspect='auto')

ax.set_title('sampl.\nsrc', size=10)
ax.set_ylim(Y[[0, -1]]+np.array([-0.5, 0.5]))
ax.invert_yaxis()

ax.axis('off')

## Sequenced before
    
ax = fig.add_subplot(gs[2])
dat = strains_info.loc[S, ['seq_before','seq_spore']].replace({'seq_before':{False:0.9, True:0.1}, 'seq_spore':{False:0, True:0.4}})\
    .sum(axis=1).values.reshape(-1,1)
mask_ref = strains_info.loc[S, 'seq_before'].isna()
dat = np.ma.masked_where(mask_ref.values.reshape(-1,1), dat)
hm_seqbefore = ax.imshow(dat, vmin=0, vmax=1, cmap='binary', aspect='auto', interpolation='nearest')

for y in np.where(mask_ref)[0]:
    ax.scatter([0], [y], s=3, c='k', 
               linewidths=0.3, marker=(6,2,0), clip_on=False)

ax.set_ylim(Y[[0, -1]]+np.array([-0.5, 0.5]))
ax.axis('off')
ax.invert_yaxis()

ax.set_title('prev.\nseq', size=10)

## Admixture

ax = fig.add_subplot(gs[3])
k = 29

S_adm = list(strains_info.loc[strains_info['species']=='cerevisiae'].index)
S_adm_idx = dict(zip(S_adm, range(422)))
S_idx = [S_adm_idx[s] for s in S]
admQ = ADM_Q[k]
admQ_cumul = np.cumsum(admQ, axis=1)
cmap = [colorcet.glasbey_category10[i+50] for i in range(k)]

for i in range(k):
    ax.barh(range(422), admQ_cumul.loc[S_idx, i], height=1, color=cmap[i], zorder=np.abs(k-i))

ax.set_xmargin(0)
ax.set_ymargin(0)

ax.set_ylim(Y[[0, -1]]+np.array([-0.5, 0.5]))
ax.invert_yaxis()
ax.axis('off')

ax.set_title(f'ADMIXTURE\n(k={k})', size=10)

## Ploidy 

ax = fig.add_subplot(gs[4])
dat = pd.Series(np.repeat(np.nan, len(S)), index=S)

for s in S:
    if s in strains_info.index:
        dat.loc[s] = strains_info.loc[s, 'ploidy']
    elif s in strains_1011.index:
        dat.loc[s] = strains_info.loc[s, 'Ploidy']
dat = pd.DataFrame(dat)

hm_ploidy = ax.imshow(dat, vmin=0, vmax=4, cmap='Greens', aspect='auto', interpolation='nearest')

ax.set_xticks([])
ax.set_yticks([])

ax.axis('off')

ax.set_title('ploidy', size=10)


## Heterozygosity

ax = fig.add_subplot(gs[5])

dat = (strains_info.loc[S, 'het_freq']*1e4).values.reshape(-1,1)
# per 10kb
het_hm = ax.imshow(dat, cmap='magma', interpolation='nearest', aspect='auto')
ax.set_title('het.\nfreq', size=10)

ax.set_ylim(Y[[0, -1]]+np.array([-0.5, 0.5]))
ax.invert_yaxis()

ax.axis('off')

## Legends

leg_x_align = 0.83

legend_elms_src = [Line2D([0], [0], color='w', marker='s', ms=12, label=label, mfc=ss)
               for label, ss in source_simplified_color.items()]
legend_src = plt.legend(handles=legend_elms_src, bbox_to_anchor=[leg_x_align, 0.75],
                        title='sampling\nsource', alignment='left',
                        loc=3, frameon=False, bbox_transform=fig.transFigure)

legend_elms_seq = [Line2D([0], [0], color='w', marker='s', ms=12, label=label, mfc=plt.get_cmap('binary')(sb)) 
               for sb, label in zip([0.1, 0.5, 0.9], ['native', 'as spore', 'never'])]
legend_seq = plt.legend(handles=legend_elms_seq, bbox_to_anchor=[leg_x_align, 0.57], 
                        title='previously\nsequenced', alignment='left',
                        loc=3, frameon=False, bbox_transform=fig.transFigure)

legend_elms_ploidy = [Line2D([0], [0], c='w', marker='s', ms=12, mfc=colormaps.get_cmap('Greens')(i/4), label=f'{i}n') for i in range(1,5)]
legend_ploidy = plt.legend(handles=legend_elms_ploidy, bbox_to_anchor=[leg_x_align, 0.37], 
                           title='ploidy', alignment='left',
                           loc=3, frameon=False, bbox_transform=fig.transFigure)

fig.add_artist(legend_src)
fig.add_artist(legend_seq)
fig.add_artist(legend_ploidy)

ax_cbar = fig.add_axes([leg_x_align+0.02, 0.15, 0.015, 0.11])
cb = plt.colorbar(het_hm, cax=ax_cbar, orientation='vertical')
cb.outline.set_visible(False)
cb_y_ticks = np.linspace(0, 63, 4)
ax_cbar.set_yticks(cb_y_ticks)
ax_cbar.set_yticklabels([f'{y:.0f}' for y in cb_y_ticks])
fig.text(leg_x_align+0.015, 0.27, 'heterozygosity\nfreq (10 kb$^{-1}$)', size=10, ha='left', va='bottom')

fig.text(0.01, 0.96, 'A', size=24, fontweight='bold')

#for ext in ['png', 'svg']:
#    plt.savefig(f'{fig_path}FigS1A.{ext}', dpi=300)

#plt.show()
plt.close()

## Fig 1DE

In [ ]:
tree_raxml_raw_path = f'{base_dir}raxml/GTGTR4_outgroup/orf_coding.random500.concat.fasta.raxml.support'
tree_raxml_raw =  Phylo.read(tree_raxml_raw_path, 'newick')
sc_tree = tree_raxml_raw.common_ancestor(['YJM333', 'YJM1321'])
#Phylo.write(sc_tree, f'{base_dir}phylo/subclades/wine_clade.nwk', format='newick')

In [ ]:
fig = plt.figure(figsize=[11, 6])
gs = plt.GridSpec(ncols=7, nrows=1, width_ratios=[10,1,1,1,6,10,10], 
                  wspace=0.18, top=0.88, left=0.02, right=0.98, bottom=0.16)

# Plot tree

ax = fig.add_subplot(gs[0])

tree_subclade = TreeViz(f'{base_dir}phylo/subclades/wine_clade.nwk', 
                        leaf_label_xmargin_ratio=0.03, leaf_label_size=4)
S = tree_subclade.leaf_labels
Y = np.arange(len(S))

tree_subclade.set_node_line_props('N_1', lw=1.5, color='k', descendent=True)

# Annotate tips

for s in S:
    clade, source = strains_info.loc[s, ['clade', 'source_simplified']]
    #tree_subclade.marker(s, marker='o', linewidths=0, alpha=1, size=4, color=clade_color[clade])
    tree_subclade.set_node_label_props(s, size=5, color='k')

tree_subclade.highlight(['YJM947', 'YJM955'], '0.7', area='full')
tree_subclade.plotfig(ax=ax)

scale_bar_dx = np.array([0, 0.001])+0.001
ax.plot(scale_bar_dx, [-3,-3], lw=2, c='k', clip_on=False)
ax.text(scale_bar_dx.mean(), -3.5, '0.001', ha='center', va='top')

ax.set_ylim(0.5, len(S)+0.5)
ax.set_xlim(0, 0.0032)

## Isolation source

ax = fig.add_subplot(gs[1])

dat = strains_info.loc[S, 'source'].apply(lambda x: source_simplified_dict[x]).values.reshape(-1,1)
ax.imshow(dat, cmap=cmap_source_simplified_color, interpolation='nearest', aspect='auto')

ax.set_title('sampl.\nsrc', size=10, rotation=90)
ax.set_ylim(Y[[0, -1]]+np.array([-0.5, 0.5]))
ax.invert_yaxis()

ax.axis('off')

## Sequenced before
    
ax = fig.add_subplot(gs[2])

dat = strains_info.loc[S, ['seq_before','seq_spore']].replace({'seq_before':{False:0.9, True:0.1}, 'seq_spore':{False:0, True:0.4}})\
    .sum(axis=1).values.reshape(-1,1)
mask_ref = strains_info.loc[S, 'seq_before'].isna()
dat = np.ma.masked_where(mask_ref.values.reshape(-1,1), dat)
hm_seqbefore = ax.imshow(dat, vmin=0, vmax=1, cmap='binary', aspect='auto', interpolation='nearest')

for y in np.where(mask_ref)[0]:
    ax.scatter([0], [y], s=10, c='k', 
               linewidths=0.3, marker=(6,2,0), clip_on=False)

ax.set_ylim(Y[[0, -1]]+np.array([-0.5, 0.5]))
ax.axis('off')
ax.invert_yaxis()

ax.set_title('prev.\nseq', size=10, rotation=90)

## Ploidy

ax = fig.add_subplot(gs[3])

dat = strains_info.loc[S, ['ploidy']].values.reshape(-1,1).astype(float)

hm_ploidy = ax.imshow(dat, vmin=0, vmax=4, cmap='Greens', aspect='auto', interpolation='nearest')

ax.set_ylim(Y[[0, -1]]+np.array([-0.5, 0.5]))
ax.invert_yaxis()
ax.axis('off')
ax.set_title('ploidy', rotation=90, size=10)

# chromosome CN

ax = fig.add_subplot(gs[5])

# horizontal cbar ax
width, height = (0.1, 0.02)
top_padding = 1.05
(xmin, ymin), (xmax, ymax) = np.array(ax.get_position())
x = np.mean(np.array([xmin, xmax])-0.5*width)
y = ymax*top_padding
ax_cbar = fig.add_axes([x, y, width, height])

dat = DP_CHROM.pivot_table(index='strain', columns='chrom', values='chrom_cn').loc[S]


sns.heatmap(dat, cmap='viridis', ax=ax, vmin=1, vmax=5,
           cbar_ax=ax_cbar, cbar_kws=dict(orientation='horizontal'))

ax_cbar.set_xticks(np.linspace(1, 5, 5))
ax_cbar.text(1.1, 0.5, 'CN', ha='left', va='center', transform=ax_cbar.transAxes)

ax.set_ylabel('')
ax.set_yticks(Y+0.5)
ax.set_yticklabels(S, size=5)
ax.set_xlabel('')
X = np.arange(16)+0.5
ax.set_xticks(X)
ax.set_xticklabels([chr_ctg_to_alias[ctg][3:] for ctg in dat.columns], size=10, rotation=90)

for yt in ax.get_yticklabels():
    s = yt.get_text()
    if s in wine_subclade + ['ADD', 'BMI', 'BMK']:
        yt.set_fontweight('bold')
        yt.set_backgroundcolor('0.8')
        yt.get_bbox_patch().set_boxstyle('square', pad=0.05)

# Repeats CN

ax = fig.add_subplot(gs[6])

# horizontal cbar ax
width, height = (0.1, 0.02)
top_padding = 1.05
(xmin, ymin), (xmax, ymax) = np.array(ax.get_position())
x = np.mean(np.array([xmin, xmax])-0.5*width)
y = ymax*top_padding
ax_cbar = fig.add_axes([x, y, width, height])

repeats_order_std = DP_REP.loc[DP_REP['repeat']!='ref|NC_001224|']\
    .pivot_table(index='strain', columns='repeat', values='cn').loc[S]\
    .std().sort_values()

dat = DP_REP.pivot_table(index='strain', columns='repeat', values='cn').loc[S, repeats_order_std.index]

sns.heatmap(dat, cmap='viridis', ax=ax,
           cbar_ax=ax_cbar, cbar_kws=dict(orientation='horizontal'))

ax_cbar.set_xticks(np.linspace(0, 150, 4))
ax_cbar.text(1.1, 0.5, 'CN', ha='left', va='center', transform=ax_cbar.transAxes)

ax.set_ylabel('')
ax.set_yticks([])
ax.set_xlabel('')
X = np.arange(repeats_order_std.shape[0])+0.5
ax.set_xticks(X)
ax.set_xticklabels([repeats_alias[r] for r in repeats_order_std.index], size=10, rotation=90)


### Legends

leg_x_align = 0.37


def hyphenate(x):
    if x == 'domesticated':
        return 'domesti-\ncated'
    elif x == 'environmental':
        return 'environ-\nmental'
    else:
        return x

legend_elms_src = [Line2D([0], [0], color='w', marker='s', ms=12, label=hyphenate(label), mfc=ss)
               for label, ss in source_simplified_color.items()]
legend_src = plt.legend(handles=legend_elms_src, bbox_to_anchor=[leg_x_align, 0.6],
                        title='sampling\nsource', alignment='left',
                        loc=3, frameon=False, bbox_transform=fig.transFigure)

legend_elms_seq = [Line2D([0], [0], color='w', marker='s', ms=12, label=label, mfc=plt.get_cmap('binary')(sb)) 
               for sb, label in zip([0.1, 0.5, 0.9], ['native', 'as spore', 'never'])]
legend_seq = plt.legend(handles=legend_elms_seq, bbox_to_anchor=[leg_x_align, 0.37], 
                        title='previously\nsequenced', alignment='left',
                        loc=3, frameon=False, bbox_transform=fig.transFigure)

legend_elms_ploidy = [Line2D([0], [0], c='w', marker='s', ms=12, mfc=colormaps.get_cmap('Greens')(i/4), label=f'{i}n') for i in range(1,5)]
legend_ploidy = plt.legend(handles=legend_elms_ploidy, bbox_to_anchor=[leg_x_align, 0.15], 
                           title='ploidy', alignment='left',
                           loc=3, frameon=False, bbox_transform=fig.transFigure)


fig.add_artist(legend_src)
fig.add_artist(legend_seq)
fig.add_artist(legend_ploidy)

fig.text(0.02, 0.93, 'D', size=24, fontweight='bold')
fig.text(0.48, 0.93, 'E', size=24, fontweight='bold')

for ext in ['png', 'svg']:
    plt.savefig(f'{fig_path}Fig1DE.{ext}', dpi=300)

plt.show()
plt.close()

In [ ]:
cyto_events = pd.read_csv('/home/mathieu/postdoc_heasley/experiments/ploidy/wine_subclade_spores/cyto_events.csv')

## Fig 1F-H

In [ ]:
fig = plt.figure(figsize=[11, 3])
gs = plt.GridSpec(ncols=4, nrows=1, width_ratios=[3,3,1,3],
                  wspace=0.35, top=0.87, left=0.08, right=0.98, bottom=0.16)

ax = fig.add_subplot(gs[0])
S = wine_subclade_order
dat = DP_REP.pivot_table(index='strain', columns='repeat', values='cn').loc[S, 'Y_prime_element.cons_trim']

ax.barh(range(10), dat, color=[strain_yprime_color[s] for s in S])
ax.set_yticks(np.arange(len(S)))
ax.set_yticklabels(S, size=9)
ax.set_xlabel('Y\' CN')
ax.invert_yaxis()

ax = fig.add_subplot(gs[1])

def rev_flog(x, T=1e9, M=4):
    return T * 10**(M*(x-1))

for s in S:
    df = cyto_events.loc[cyto_events['strain']==s].copy()
    df['raw'] = df['FL03-A'].apply(rev_flog)
    df['log'] = np.log10(df['raw'])
    d = df['log'][df['singlets']].sort_values()
    c = strain_yprime_color[s]
    sns.kdeplot(d, color=c, lw=2, label=s)

ax.legend(loc=3, bbox_to_anchor=[1, 0], frameon=False)

ax.set_ylabel('density')
ax.set_xlabel('log SYTOX Green (AU)')
ax.set_xlim(8, 8.8)

ax = fig.add_subplot(gs[3])

dat = pd.concat([DP_REP.loc[(DP_REP['strain'].isin(wine_subclade)) & (DP_REP['repeat']=='Y_prime_element.cons_trim')], 
                 DP_REP_STROPE.loc[(DP_REP_STROPE['repeat']=='Y_prime_element.cons_trim')]], axis=0)\
.merge(strains_info_strope.reset_index(drop=True), on='strain', how='left')
dat['background'] = np.where(dat['parent'].isna(), dat['strain'], dat['parent'])
dat['type'] = np.where(dat['background']==dat['strain'], 'native', 'spore')
dat = dat.pivot_table(index='background', columns='type', values='cn', aggfunc=lambda x: x)

ax.plot([0, 200], [0,200], lw=1, c='0.5', zorder=1)
ax.scatter(dat['native'], dat['spore'], c=[strain_yprime_color[s] for s in dat.index], zorder=1)

ax.set_xlabel('native isolate')
ax.set_ylabel('spore derivative')

fig.text(0.02, 0.88, 'F', size=24, fontweight='bold')
fig.text(0.3, 0.88, 'G', size=24, fontweight='bold')
fig.text(0.7, 0.88, 'H', size=24, fontweight='bold')

sns.despine()

for ext in ['png', 'svg']:
    plt.savefig(f'{fig_path}Fig1F-H.{ext}', dpi=300)

plt.show()
plt.close()

# Summary of individual strains

In [ ]:
idx = 0

with ProgressBar(max_value=len(vcf_samples_s)) as bar:

    #for s in vcf_samples_s:
    for s in ['YJM181']:
        
        s_idx = vcf_samples_s_idx[s]
        
        fig = plt.figure(figsize=[18,7])
        gs = plt.GridSpec(ncols=16, nrows=4, width_ratios=tig_off['len'].iloc[:-1], height_ratios=[2,2,2,3],
                          hspace=0.3, wspace=0.15, left=0.08, right=0.93, top=0.92, bottom=0.1)
        
        chr_ax = dict(zip(chrom_order, range(17)))
        
        # DEPTH
        dat = DEPTH_NUC.loc[(DEPTH_NUC['strain']==s) & (DEPTH_NUC['CHROM'].isin(nuclear_chromosomes))].copy()
        dat['dp_norm'] = dat['DP']/dat['DP'].median()
        ylim = (0, np.quantile(dat['dp_norm'], 0.98)*1.3)
        
        for ctg, df in dat.groupby('CHROM'):
            chr = chr_ctg_to_alias[ctg]
            ax = fig.add_subplot(gs[0, chr_ax[chr]])
            
            #ax.scatter(df['POS'], df['dp_norm'], c='darkblue', s=4)
            ax.fill_between(df['POS'], df['dp_norm'].astype(np.float64), color='dodgerblue', zorder=1)
            ax.axhline(1, ls='--', lw=1, c='k', zorder=2)
            
            yticks = np.linspace(0, ylim[1], 5)
            ax.set_ylim(ylim)
            ax.set_yticks(yticks)
            if chr_ax[chr] == 0:
                ax.set_yticklabels([f'{y:.1f}' for y in yticks])
                ax.set_ylabel('Relative depth\nof coverage', c='dodgerblue')
            else:
                ax.set_yticklabels([])
            
            xticks = np.arange(0, tig_off.loc[chr, 'len'], 1e5)
            ax.set_xlim(0, tig_off.loc[chr, 'len'])
            ax.set_xticks(xticks)
            ax.set_xticklabels([])
            ax.set_title(chr[3:])
            ax.axvline(centromeres.loc[ctg], color='k', lw=0.5)
        
        
        # SNP density
        dat = SNP_WIN.loc[(SNP_WIN['strain']==s) & (SNP_WIN['ctg'].isin(nuclear_chromosomes))].copy()
        dat['snp_freq'] = dat['snp_freq']*1e4
        ylim = (0, np.quantile(dat['snp_freq'], 0.98)*1.3)
        
        for ctg, df in dat.groupby('ctg'):
            chr = chr_ctg_to_alias[ctg]
            ax = fig.add_subplot(gs[1, chr_ax[chr]])
        
            #ax.scatter(df['pos_bin'], df['snp_freq'], c='r', s=4)
            ax.fill_between(df['pos_bin'], df['snp_freq'].astype(np.float64), color='firebrick')
            
            yticks = np.linspace(0, ylim[1], 5)
            ax.set_ylim(ylim)
            ax.set_yticks(yticks)
            if chr_ax[chr] == 0:
                ax.set_yticklabels([f'{y:.1f}' for y in yticks])
                ax.set_ylabel('Homozygous SNPs\nper 10 kb', c='firebrick')
            else:
                ax.set_yticklabels([])
            
            xticks = np.arange(0, tig_off.loc[chr, 'len'], 1e5)
            ax.set_xlim(0, tig_off.loc[chr, 'len'])
            ax.set_xticks(xticks)
            ax.set_xticklabels([])
            ax.axvline(centromeres.loc[ctg], color='k', lw=0.5)
        
        # HET snp density
        dat = HET_WIN.loc[(HET_WIN['strain']==s) & (HET_WIN['ctg'].isin(nuclear_chromosomes))].copy()
        dat['het_freq'] = dat['het_freq']*1e4
        ylim = (0, np.quantile(dat['het_freq'], 0.98)*1.3)
        
        for ctg, df in dat.groupby('ctg'):
            chr = chr_ctg_to_alias[ctg]
            ax = fig.add_subplot(gs[2, chr_ax[chr]])
        
            #ax.scatter(df['pos_bin'], df['het_freq'], c='limegreen', s=4)
            ax.fill_between(df['pos_bin'], df['het_freq'].astype(np.float64), color='limegreen')
            
            yticks = np.linspace(0, ylim[1], 5)
            ax.set_ylim(ylim)
            ax.set_yticks(yticks)
            if chr_ax[chr] == 0:
                ax.set_yticklabels([f'{y:.1f}' for y in yticks])
                ax.set_ylabel('Heterozygous SNPs\nper 10 kb', c='limegreen')
            else:
                ax.set_yticklabels([])
        
            xticks = np.arange(0, tig_off.loc[chr, 'len'], 1e5)
            ax.set_xlim(0, tig_off.loc[chr, 'len'])
            ax.set_xticks(xticks)
            ax.set_xticklabels([])
            ax.axvline(centromeres.loc[ctg], color='k', lw=0.5)
        
        # MAF heatmap
        dat = pd.DataFrame(np.concatenate([CHROM_s.reshape(-1,1), POS_s.reshape(-1,1), maf[:, s_idx].reshape(-1,1)], axis=1))
        dat = dat[gt_s_het[:, s_idx]]
        dat = dat.loc[dat[0].isin(nuclear_chromosomes)]
        dat['chrom'] = dat[0].apply(lambda x: chr_ctg_to_alias[x])
        dat['pos_abs'] = dat[1] + tig_off.loc[dat['chrom'], 'cumul_start'].values
        vmax = np.quantile(np.histogram2d(dat[2], dat['pos_abs'], bins=[np.linspace(0.5,1,51), np.arange(0, 12.2e6, 1e4)])[0], 0.98)
        if vmax == 0:
            vmax = 1
        
        for ctg, df in dat.groupby(0):
            chr = chr_ctg_to_alias[ctg]
            ax = fig.add_subplot(gs[3, chr_ax[chr]])
            
            hist = np.histogram2d(df[2], df[1], bins=[np.linspace(0.5,1,51), np.arange(0, tig_off.loc[chr, 'len']+1e4, 1e4)])
            hm = ax.imshow(hist[0], cmap='viridis', interpolation='nearest', aspect='auto', vmin=0, vmax=vmax)
        
            yticks = np.linspace(0, 50, 5)
            yticks = np.array([0.5, 0.6777, 0.75, 0.8333, 1])
            ax.set_ylim(-0.5, 49.5)
            ax.set_yticks((yticks-0.5)*100)
            if chr_ax[chr] == 0:
                ax.set_yticklabels([f'{y:.3f}' for y in yticks])
                ax.set_ylabel('Heterozygous SNPs\nmajor allele freq')
            else:
                ax.set_yticklabels([])
        
            xticks = np.arange(0, tig_off.loc[chr, 'len'], 1e5)/1e4
            ax.set_xlim(0, tig_off.loc[chr, 'len']/1e4)
            ax.set_xticks(xticks)
            ax.set_xticklabels(xticks*10, rotation=90, size=7)
            ax.axvline(centromeres.loc[ctg]/1e4, color='red', lw=0.5)
        
        ax_cbar = fig.add_axes([0.94, 0.08, 0.01, 0.17])
        cb = plt.colorbar(hm, cax=ax_cbar, orientation='vertical')
        cb.outline.set_visible(False)
        cb_y_ticks = np.linspace(0, vmax, 5)
        ax_cbar.set_yticks(cb_y_ticks)
        ax_cbar.set_yticklabels([f'{x:.1f}' for x in cb_y_ticks])
        ax_cbar.text(0, 1.15, 'Het SNPs\nper 10 kb', 
                     ha='left', va='bottom', transform=ax_cbar.transAxes)
        
        fig.text(0.5, 0.99, s, size=14, ha='center', va='top')
        fig.text(0.5, 0.02, 'position (kb)')
        
        sns.despine()
        plt.savefig(f'{fig_path}summary_indiv_strains/{s}.summary.png', dpi=300)
        #plt.show()
        plt.close()

        idx += 1
        bar.update(idx)